Cài đặt thư viện

In [ ]:
!pip install web3==6.11.3 eth-account==0.10.0 gradio requests py-solc-x

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of eth-rlp to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 81.3 MB/s eta 0:00:00
  Created wheel for lru-dict: filename=lru_dict-1.2.0-cp312-cp312-linux_x86_64.whl size=28684 sha256=66f822dd

Upload file config.txt

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

with open("config.txt") as f:
    for line in f:
        if "=" in line:
            k, v = line.strip().split("=", 1)
            os.environ[k] = v

print("Đã nạp cấu hình thành công!")

Saving config.txt to config.txt
Đã nạp cấu hình thành công!


Kết nối blockchain

In [ ]:
from web3 import Web3
from eth_account import Account
import os

INFURA_URL = os.environ["INFURA_URL"]
PRIVATE_KEY = os.environ["PRIVATE_KEY"]

w3 = Web3(Web3.HTTPProvider(INFURA_URL))
acct = Account.from_key(PRIVATE_KEY)

print("Kết nối:", w3.is_connected())
print("Ví:", acct.address)
print("Số dư ETH:", w3.from_wei(w3.eth.get_balance(acct.address), "ether"))

Kết nối: True
Ví: 0x03be9eeE867458F5FC5b65Ff07C647E2B5b81C6a
Số dư ETH: 0.071824113560542028


Tạo file hợp đồng CoffeeTrace.sol

In [ ]:
%%writefile CoffeeTrace.sol
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract CoffeeTrace {

    address public owner;
    bool public paused;

    struct Batch {
        string batchId;
        string lotHash;
        string origin;
        uint256 weightKg;
        address currentOwner;
        uint256 createdAt;
        bool exists;
        bool archived;
    }

    struct QualityCert {
        string qcId;
        string batchId;
        string qcHash;
        string inspector;
        uint256 timestamp;
        bool exists;
    }

    struct Shipment {
        string shipId;
        string batchId;
        string blHash;
        string destination;
        uint256 shipDate;
        bool exists;
    }

    mapping(string => Batch) private batches;
    mapping(string => QualityCert) private qcs;
    mapping(string => Shipment) private shipments;

    string[] private batchIndex;

    event BatchCreated(string indexed batchId, address indexed creator, string lotHash);
    event BatchUpdated(string indexed batchId, address indexed updater, string newLotHash);
    event BatchTransferred(string indexed batchId, address indexed from, address indexed to);
    event BatchArchived(string indexed batchId, address actor, string reason);

    event QCCreated(string indexed qcId, string indexed batchId, string qcHash);
    event ShipmentCreated(string indexed shipId, string indexed batchId, string blHash);

    event ContractPaused(address actor);
    event ContractUnpaused(address actor);
    event ContractDestroyed(address actor, uint256 balance);

    modifier onlyOwner() { require(msg.sender == owner, "Only owner"); _; }
    modifier existsBatch(string memory _batchId) { require(batches[_batchId].exists, "Batch not exists"); _; }
    modifier notPaused() { require(!paused, "Paused"); _; }

    constructor() { owner = msg.sender; paused = false; }

    function pause() external onlyOwner {
        paused = true;
        emit ContractPaused(msg.sender);
    }

    function unpause() external onlyOwner {
        paused = false;
        emit ContractUnpaused(msg.sender);
    }

    function transferOwnership(address _newOwner) external onlyOwner {
        require(_newOwner != address(0), "Zero");
        owner = _newOwner;
    }

    function createBatch(
        string calldata _batchId,
        string calldata _lotHash,
        string calldata _origin,
        uint256 _weightKg,
        address _initialOwner
    ) external notPaused {
        require(!batches[_batchId].exists, "Exists");
        require(_initialOwner != address(0), "Invalid");

        batches[_batchId] = Batch({
            batchId: _batchId,
            lotHash: _lotHash,
            origin: _origin,
            weightKg: _weightKg,
            currentOwner: _initialOwner,
            createdAt: block.timestamp,
            exists: true,
            archived: false
        });

        batchIndex.push(_batchId);

        emit BatchCreated(_batchId, msg.sender, _lotHash);
    }

    function updateBatchHash(string calldata _batchId, string calldata _newLotHash)
        external existsBatch(_batchId) notPaused
    {
        batches[_batchId].lotHash = _newLotHash;
        emit BatchUpdated(_batchId, msg.sender, _newLotHash);
    }

    function transferBatch(string calldata _batchId, address _to)
        external existsBatch(_batchId) notPaused
    {
        require(_to != address(0), "Invalid");
        Batch storage b = batches[_batchId];
        require(msg.sender == b.currentOwner || msg.sender == owner, "Unauthorized");

        address previous = b.currentOwner;
        b.currentOwner = _to;

        emit BatchTransferred(_batchId, previous, _to);
    }

    function archiveBatch(string calldata _batchId, string calldata _reason)
        external existsBatch(_batchId) notPaused
    {
        batches[_batchId].archived = true;
        emit BatchArchived(_batchId, msg.sender, _reason);
    }

    function addQualityCert(
        string calldata _qcId,
        string calldata _batchId,
        string calldata _qcHash,
        string calldata _inspector
    ) external existsBatch(_batchId) notPaused {
        require(!qcs[_qcId].exists, "QC exists");

        qcs[_qcId] = QualityCert({
            qcId: _qcId,
            batchId: _batchId,
            qcHash: _qcHash,
            inspector: _inspector,
            timestamp: block.timestamp,
            exists: true
        });

        emit QCCreated(_qcId, _batchId, _qcHash);
    }

    function createShipment(
        string calldata _shipId,
        string calldata _batchId,
        string calldata _blHash,
        string calldata _destination,
        uint256 _shipDate
    ) external existsBatch(_batchId) notPaused {
        require(!shipments[_shipId].exists, "Ship exists");

        shipments[_shipId] = Shipment({
            shipId: _shipId,
            batchId: _batchId,
            blHash: _blHash,
            destination: _destination,
            shipDate: _shipDate,
            exists: true
        });

        emit ShipmentCreated(_shipId, _batchId, _blHash);
    }

    function getBatch(string calldata _batchId)
        external view existsBatch(_batchId)
        returns (Batch memory)
    {
        return batches[_batchId];
    }

    function getQC(string calldata _qcId) external view returns (QualityCert memory) {
        require(qcs[_qcId].exists, "Not exists");
        return qcs[_qcId];
    }

    function getShipment(string calldata _shipId) external view returns (Shipment memory) {
        require(shipments[_shipId].exists, "Not exists");
        return shipments[_shipId];
    }

    function listBatchIds() external view returns (string[] memory) {
        return batchIndex;
    }

    function withdraw(address payable _to) external onlyOwner {
        require(_to != address(0), "Zero");
        uint256 bal = address(this).balance;
        require(bal > 0, "No balance");
        (bool sent,) = _to.call{value: bal}("");
        require(sent, "Failed");
    }

    function destroy(address payable _to) external onlyOwner {
        emit ContractDestroyed(msg.sender, address(this).balance);
        selfdestruct(_to);
    }

    receive() external payable {}
}

Writing CoffeeTrace.sol


Cài solc và biên dịch hợp đồng (Local)

In [ ]:
import hashlib
import time

data_to_hash = "Batch-001 updated with new processing details" + str(time.time())
encoded_data = data_to_hash.encode('utf-8')
new_hash = hashlib.sha256(encoded_data).hexdigest()

print("Dữ liệu gốc:", data_to_hash)
print("Hash mới:", new_hash)

Dữ liệu gốc: Batch-001 updated with new processing details1764745215.8244326
Hash mới: e3021ab5585add41fa96b5f0cb01b38ea0359b9c6efc2139bbc97a3d0cb59491


In [ ]:
from solcx import install_solc, set_solc_version, compile_standard
import json

install_solc("0.8.20")
set_solc_version("0.8.20")

with open("CoffeeTrace.sol", "r") as f:
    source = f.read()

compiled = compile_standard({
    "language": "Solidity",
    "sources": {"CoffeeTrace.sol": {"content": source}},
    "settings": {"outputSelection": {"*": {"*": ["abi", "evm.bytecode"]}}}
}, solc_version="0.8.20")

abi = compiled["contracts"]["CoffeeTrace.sol"]["CoffeeTrace"]["abi"]
bytecode = compiled["contracts"]["CoffeeTrace.sol"]["CoffeeTrace"]["evm"]["bytecode"]["object"]

with open("CoffeeTraceABI.json", "w") as f:
    json.dump(abi, f)

print(" Compile thành công. ABI lưu CoffeeTraceABI.json")

 Compile thành công. ABI lưu CoffeeTraceABI.json


Deploy hợp đồng CoffeeTrace lên Sepolia

In [ ]:
Coffee = w3.eth.contract(abi=abi, bytecode=bytecode)

nonce = w3.eth.get_transaction_count(acct.address, "pending")
gas_est = Coffee.constructor().estimate_gas({"from": acct.address})
print("Estimated gas:", gas_est)

tx = Coffee.constructor().build_transaction({
    "from": acct.address,
    "nonce": nonce,
    "gas": gas_est + 200000,
    "gasPrice": w3.to_wei("1", "gwei"),
    "chainId": w3.eth.chain_id
})

signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
tx_hash = w3.eth.send_raw_transaction(signed.rawTransaction)
print("Đang deploy... TX hash:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=600)
contract_address = receipt.contractAddress
print(" Deploy thành công. Contract address:", contract_address)

Estimated gas: 3351253
Đang deploy... TX hash: 0x48f699d05a73e488e18fd824f8ac264238c6dc44af79628a80109726fb479e90
 Deploy thành công. Contract address: 0xFa6817dcfc9BCfb9e3D4eF330180a761Cd945e24


Khởi tạo Contract object và helper send_tx

In [ ]:
contract = w3.eth.contract(address=contract_address, abi=abi)

def send_tx(fn, gas=400000):
    nonce = w3.eth.get_transaction_count(acct.address, "pending")
    tx = fn.build_transaction({
        "from": acct.address,
        "nonce": nonce,
        "gas": gas,
        "gasPrice": w3.to_wei("1", "gwei"), # Đặt gasPrice thấp nhất cho các giao dịch thử nghiệm
        "chainId": w3.eth.chain_id
    })
    signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    txh = w3.eth.send_raw_transaction(signed.rawTransaction)
    return txh.hex()

Các hàm gọi Blockchain

In [ ]:
def create_batch(batchId, lotHash, origin, weightKg, initialOwner):
    return send_tx(contract.functions.createBatch(batchId, lotHash, origin, int(weightKg), initialOwner), gas=600000)

def update_batch_hash(batchId, newHash):
    return send_tx(contract.functions.updateBatchHash(batchId, newHash), gas=250000)

def transfer_batch(batchId, to_addr):
    return send_tx(contract.functions.transferBatch(batchId, to_addr), gas=250000)

def archive_batch(batchId, reason):
    return send_tx(contract.functions.archiveBatch(batchId, reason), gas=200000)

def add_qc(qcId, batchId, qcHash, inspector):
    return send_tx(contract.functions.addQualityCert(qcId, batchId, qcHash, inspector), gas=300000)

def create_shipment(shipId, batchId, blHash, destination, shipDate):
    return send_tx(contract.functions.createShipment(shipId, batchId, blHash, destination, int(shipDate)), gas=350000)

def get_batch(batchId):
    return contract.functions.getBatch(batchId).call()

Hàm xử lý UI(Gradio)

Giao diện Gradio đầy đủ

In [ ]:

import gradio as gr
import hashlib
import time
import datetime

# HÀM FORMAT THỜI GIAN
def format_timestamp(ts):
    try:
        utc = datetime.datetime.utcfromtimestamp(int(ts))
        vn = utc + datetime.timedelta(hours=7)
        return vn.strftime("%Y-%m-%d %H:%M:%S (GMT+7)")
    except:
        return f"{ts} (timestamp không hợp lệ)"


# HÀM SINH HASH
def auto_hash(*args):
    raw = "|".join([str(x) for x in args]) + "|" + str(time.time())
    return hashlib.sha256(raw.encode()).hexdigest()


# VIEW BATCH
def ui_view_batch(batchId):
    try:
        b = get_batch(batchId)
        (id_b, lotHash, origin, weightKg, ownerAddr, createdAt, exists, archived) = b
        status = "Đã lưu trữ" if archived else "Đang hoạt động"

        created_fmt = format_timestamp(createdAt)

        return (
            f"ID Lô: {id_b}\n"
            f"Hash Lô: {lotHash}\n"
            f"Nguồn gốc: {origin}\n"
            f"Trọng lượng: {weightKg} kg\n"
            f"Chủ sở hữu: {ownerAddr}\n"
            f"Ngày tạo: {created_fmt}\n"
            f"Trạng thái: {status}"
        )
    except Exception as e:
        return f"Lỗi: {e}"



# CREATE BATCH – AUTO HASH
def ui_create_batch(batchId, origin, weight, owner):
    try:
        lotHash = auto_hash(batchId, origin, weight, owner)

        tx_hash = create_batch(batchId, lotHash, origin, int(weight), owner)
        w3.eth.wait_for_transaction_receipt(tx_hash)

        return (
            f"Tạo lô thành công!\n"
            f"ID Lô: {batchId}\n"
            f"Hash Lô (tự động): {lotHash}\n"
            f"Nguồn gốc: {origin}\n"
            f"Trọng lượng: {weight}\n"
            f"Chủ sở hữu: {owner}\n"
            f"Hash Giao dịch: {tx_hash}"
        )
    except Exception as e:
        return f"Lỗi: {e}"


# UPDATE HASH (AUTO HASH FROM ADDITIONAL DATA)
def ui_update_hash(batchId, additional_data_for_hash):
    try:
        newHash = auto_hash(batchId, additional_data_for_hash)

        txh = update_batch_hash(batchId, newHash)
        w3.eth.wait_for_transaction_receipt(txh)
        return f"Cập nhật thành công! Hash mới (tự động): {newHash}. Hash Giao dịch: {txh}"
    except Exception as e:
        return f"Lỗi: {e}"


# TRANSFER OWNER
def ui_transfer(batchId, toAddr):
    try:
        txh = transfer_batch(batchId, toAddr)
        return f"Chuyển giao thành công. Hash Giao dịch: {txh}"
    except Exception as e:
        return f"Lỗi: {e}"


# QC – AUTO HASH
def ui_add_qc(qcId, batchId, inspector_name, grade, moisture):
    try:
        qc_timestamp = int(time.time())  # sinh tự động

        qcHash = auto_hash(batchId, inspector_name, grade, moisture, qc_timestamp)

        tx_hash = add_qc(qcId, batchId, qcHash, inspector_name)
        w3.eth.wait_for_transaction_receipt(tx_hash)

        qc_fmt = format_timestamp(qc_timestamp)

        return (
            f"Đã thêm Chứng nhận Chất lượng!\n"
            f"ID QC: {qcId}\n"
            f"ID Lô: {batchId}\n"
            f"Hash QC (tự động): {qcHash}\n"
            f"Người kiểm tra: {inspector_name}\n"
            f"Hạng: {grade}\n"
            f"Độ ẩm: {moisture}\n"
            f"Thời gian QC: {qc_fmt}\n"
            f"Hash Giao dịch: {tx_hash}"
        )
    except Exception as e:
        return f"Lỗi: {e}"



# SHIPMENT – AUTO HASH
def create_shipment_ui(ship_id, batch_id, origin, destination, transporter, arrival_input):
    try:
        if not arrival_input or str(arrival_input).strip() == "":
            arrival_ts = int(time.time())
        else:
            arrival_ts = int(float(arrival_input))

        shipHash = auto_hash(ship_id, batch_id, origin, destination, transporter, arrival_ts)

        func = contract.functions.createShipment(
            ship_id,
            batch_id,
            shipHash,
            destination,
            arrival_ts
        )

        txh = send_tx(func)
        arrival_fmt = format_timestamp(arrival_ts)

        return (
            f"Đã tạo Lô hàng!\n"
            f"ID Lô hàng: {ship_id}\n"
            f"Hash Lô hàng (tự động): {shipHash}\n"
            f"Nguồn gốc: {origin}\n"
            f"Điểm đến: {destination}\n"
            f"Đơn vị vận chuyển: {transporter}\n"
            f"Thời gian đến dự kiến: {arrival_fmt}\n"
            f"Hash Giao dịch: {txh}"
        )
    except Exception as e:
        return f"Lỗi: {e}"


# GRADIO UI
with gr.Blocks(title="CoffeeTrace - UI") as demo:
    gr.Markdown("## CoffeeTrace - Truy xuất nguồn gốc cà phê")

    with gr.Tab("Tra cứu"):
        b_in = gr.Textbox(label="ID Lô")
        b_out = gr.Textbox(label="Kết quả", interactive=False, lines=8)
        gr.Button("Tra cứu").click(ui_view_batch, inputs=b_in, outputs=b_out)

    with gr.Tab("Tạo lô"):
        bi = gr.Textbox(label="ID Lô")
        og = gr.Textbox(label="Nguồn gốc")
        wt = gr.Number(label="Trọng lượng (kg)", value=0)
        ow = gr.Textbox(label="Địa chỉ chủ sở hữu")
        outc = gr.Textbox(label="Kết quả", interactive=False, lines=8)

        gr.Button("Tạo lô").click(ui_create_batch, inputs=[bi, og, wt, ow], outputs=outc)

    with gr.Tab("Cập nhật Hash"):
        ub = gr.Textbox(label="ID Lô")
        # Thay đổi từ nhập hash thủ công sang nhập dữ liệu để tự động tạo hash
        additional_data_input = gr.Textbox(label="Thông tin bổ sung cho Hash mới (ví dụ: đã rang, liên kết báo cáo mới)")
        outu = gr.Textbox(label="Kết quả", lines=6)
        gr.Button("Cập nhật").click(ui_update_hash, inputs=[ub, additional_data_input], outputs=outu)

    with gr.Tab("Chuyển giao"):
        cb = gr.Textbox(label="ID Lô")
        ta = gr.Textbox(label="Địa chỉ người nhận")
        outt = gr.Textbox(label="Kết quả", lines=6)
        gr.Button("Chuyển").click(ui_transfer, inputs=[cb, ta], outputs=outt)

    with gr.Tab("Chứng nhận Chất lượng (QC)"):
        q1 = gr.Textbox(label="ID QC")
        q2 = gr.Textbox(label="ID Lô")
        inspector = gr.Textbox(label="Người kiểm tra")
        grade = gr.Textbox(label="Hạng")
        moisture = gr.Textbox(label="Độ ẩm (%)")
        outq = gr.Textbox(label="Kết quả", lines=8)

        gr.Button("Thêm QC").click(
             ui_add_qc,
             inputs=[q1, q2, inspector, grade, moisture],
             outputs=outq
        )

    with gr.Tab("Vận chuyển"):
        ship_id = gr.Textbox(label="ID Lô hàng")
        batch_for_ship = gr.Textbox(label="ID Lô")
        origin_ship = gr.Textbox(label="Nguồn gốc")
        destination = gr.Textbox(label="Điểm đến")
        transporter = gr.Textbox(label="Đơn vị vận chuyển")
        arrival_ts = gr.Textbox(label="Timestamp đến (UNIX)",
                                placeholder="Để trống để tự sinh timestamp")
        out_ship = gr.Textbox(label="Kết quả", lines=8)

        gr.Button("Tạo Lô hàng").click(
            create_shipment_ui,
            inputs=[ship_id, batch_for_ship, origin_ship, destination, transporter, arrival_ts],
            outputs=out_ship
        )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4636abecfa6c3652b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
